In [ ]:
import pandas as pd
import seaborn as sns

load_to_file = 'heart_disease_uci.csv'
df = pd.read_csv(load_to_file, delimiter=',')
print("df :", df)
print("df.head(): ", df.head())
print("df.shape; ", df.shape)


In [ ]:
#clean dataset
print("\n Missing values:" )
print(df.isnull().sum)

#filling missing values
df['trestbps'] = df['trestbps'].fillna(df['trestbps'].median())
df['chol'] = df['chol'].fillna(df['chol'].median())
df['fbs'] = df['fbs'].fillna(df['fbs'].mode()[0])
df['restecg'] = df['restecg'].fillna(df['restecg'].mode()[0])
df['thalch'] = df['thalch'].fillna(df['thalch'].median())
df['exang'] = df['exang'].fillna(df['exang'].mode()[0])
df['oldpeak'] = df['oldpeak'].fillna(df['oldpeak'].median())
df['slope'] = df['slope'].fillna(df['slope'].mode()[0])
df['ca'] = df['ca'].fillna(df['ca'].mode()[0])
df['thal'] = df['thal'].fillna(df['thal'].mode()[0])

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

#converting categorial data in numbers
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in ['sex', 'cp', 'restecg', 'slope', 'thal', 'dataset']:
    df[col] = le.fit_transform(df[col].astype(str))


#Target will change to binary 0 or 1 (0 = no disease, 1 = disease)
df['target'] = (df['num'] > 0).astype(int)

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())



In [ ]:
import matplotlib.pyplot as plt
#countplot
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=df)
plt.title('Heart Disease Distribution (0=No, 1=Yes)')
plt.show()

# Age vs Target
plt.figure(figsize=(8,5))
sns.boxplot(x='target', y='age', data=df)
plt.title('Age vs Heart Disease')
plt.show()

# Correlation heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:
# features and target
features = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
            'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

X = df[features]
y = df['target']

# train model
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Logistic Regression
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
from sklearn.metrics import accuracy_score
print("\nLogistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

# Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))

#confusion matrix
from sklearn.metrics import confusion_matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, lr_pred),
            annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ROC CURVE
from sklearn.metrics import roc_curve, auc
lr_probs = lr_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, lr_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,5))
plt.plot(fpr, tpr, color='blue', label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1], [0,1], color='gray', linestyle='--')
plt.title('ROC Curve - Logistic Regression')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

#FEATURE IMPORTANCE 
importance = pd.Series(dt_model.feature_importances_, index=features)
importance = importance.sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=importance.values, y=importance.index)
plt.title('Feature Importance - Decision Tree')
plt.show()